# Debug DSS Knowledge Bank
Creates a dataset that lists all the files referenced in a DSS knowledge bank, and lists the number of entries in the vector store for each one

Set the Python Env as INTERNAL_retrieval_augmented_generation

DSS v13.5

In [ ]:
# Replace with the knowledge bank ID for the current project.
kb_id               = "thxGWJgB"
# Replace with the name of the output dataset (must already exist).
output_dataset_name = "kb_debug"


In [ ]:
from collections import defaultdict
import json

import dataiku
import numpy as np
import pandas as pd
from dataiku import pandasutils as pdu


In [ ]:
client = dataiku.api_client()
project = client.get_default_project()

In [ ]:
kb1                      = project.get_knowledge_bank(kb_id)
kb2                      = kb1.as_core_knowledge_bank()
kb_langchain_vectorstore = kb2.as_langchain_vectorstore()
kb_langchain_retriever   = kb2.as_langchain_retriever()


In [ ]:
def count_sources_in_full_vectorstore(vectorstore) -> dict:
    """Count all occurrences of each 'source' field in the full LangChain vector store."""

    source_counts = defaultdict(int)

    # Chroma exposes docs via _collection.get(), FAISS requires custom access
    if hasattr(vectorstore, "_collection"):  # likely Chroma
        # Chroma-specific: fetch all metadata
        all_docs = vectorstore._collection.get(include=["metadatas"])
        metadatas = all_docs.get("metadatas", [])
    elif hasattr(vectorstore, "index_to_docstore"):  # likely FAISS or similar
        metadatas = [
            doc.metadata for doc in vectorstore.index_to_docstore.values()
        ]
    else:
        raise NotImplementedError("This vectorstore backend is not directly supported.")

    # Count sources
    for metadata in metadatas:
        x = metadata.get("DKU_DOCUMENT_INFO", "source_file")
        source = json.loads(x)['source_file']['path']
        source_counts[source] += 1

    return dict(source_counts)

In [ ]:
full_counts = count_sources_in_full_vectorstore(kb_langchain_vectorstore)

In [ ]:
output_df = pd.DataFrame(
    list(full_counts.items()), 
    columns=['path', 'num_vector_store_entries']
)

In [ ]:
output_df.sort_values(by='num_vector_store_entries', ascending=False, inplace=True)

In [ ]:
output_dataset        = dataiku.Dataset(output_dataset_name)
output_dataset.write_with_schema(output_df)